In [1]:
%%capture
!pip install unsloth datasets scikit-learn pandas accelerate

In [1]:
import json
import re
import torch
import numpy as np
import pandas as pd
from typing import Iterator, Dict

from datasets import Dataset
from sklearn.model_selection import KFold
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_NAME      = "unsloth/Llama-3.2-1B-Instruct"
MAX_SEQ_LENGTH  = 512
LOAD_IN_4BIT    = True

# ── LoRA ─────────────────────────────────────────────────────────────────────
LORA_R          = 16
LORA_ALPHA      = 16
LORA_DROPOUT    = 0.0
TARGET_MODULES  = ["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"]

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS          = 1
BATCH_SIZE      = 64
GRAD_ACCUM      = 1
LR              = 2e-4
WARMUP_RATIO    = 0.05
OUTPUT_DIR      = "./outputs"

# ── Splits ───────────────────────────────────────────────────────────────────
SEED:      int   = 42
N_FOLDS:   int   = 3

# ── System prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "Your task is to extract address components from the user's message. "
    "Ignore any missing information and exclude those fields from your output. "
    "Return the data in JSON format using this structure:\n"
    "Output format:\n"
    "{\n"
    '  "house_number": "<house number>",\n'
    '  "street": "<street name>",\n'
    '  "city": "<city name>",\n'
    '  "postal_code": "<postal or zip code>",\n'
    '  "state": "<state or province name>",\n'
    '  "country": "<country name>"\n'
    "}"
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


### Dataset

In [2]:
from data_preparation.preprocess_data import make_splits, preprocess_data

df = preprocess_data()

Loading dataset...
Cleaning data...


### Chat-template formatting

In [3]:
def df_to_hf_dataset(df: pd.DataFrame, tokenizer) -> Dataset:
    """Convert DataFrame rows → formatted chat strings via the tokenizer chat template."""

    def _format(row):
        # Drop empty fields from target
        target = {k: v for k, v in row["target"].items() if v}
        messages = [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": row["input"]},
            {"role": "assistant", "content": json.dumps(target, ensure_ascii=False)},
        ]
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

    texts = df.apply(_format, axis=1).tolist()
    return Dataset.from_dict({"text": texts})

### Training loop over folds

In [4]:
fold_results = []

for fold_idx, split in enumerate(make_splits(df)):
    print(f"\n{'='*60}")
    print(f" FOLD {fold_idx + 1} / {N_FOLDS}")
    print(f"  train={len(split['train'])}  val={len(split['val'])}  test={len(split['test'])}")
    print(f"{'='*60}")

    # ── Load model & tokenizer (fresh each fold) ──────────────────────────────
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = MODEL_NAME,
        max_seq_length = MAX_SEQ_LENGTH,
        load_in_4bit   = LOAD_IN_4BIT,
        dtype          = None,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r              = LORA_R,
        target_modules = TARGET_MODULES,
        lora_alpha     = LORA_ALPHA,
        lora_dropout   = LORA_DROPOUT,
        bias           = "none",
        use_gradient_checkpointing = "unsloth",
        random_state   = SEED,
    )

    train_ds = df_to_hf_dataset(split["train"], tokenizer)
    val_ds   = df_to_hf_dataset(split["val"],   tokenizer)

    fold_output_dir = f"{OUTPUT_DIR}/fold_{fold_idx + 1}"

    trainer = SFTTrainer(
        model          = model,
        tokenizer      = tokenizer,
        train_dataset  = train_ds,
        eval_dataset   = val_ds,
        args           = SFTConfig(
            output_dir                  = fold_output_dir,
            num_train_epochs            = EPOCHS,
            per_device_train_batch_size = 64,
            per_device_eval_batch_size  = 32,
            gradient_accumulation_steps = GRAD_ACCUM,
            learning_rate               = LR,
            warmup_ratio                = WARMUP_RATIO,
            lr_scheduler_type           = "cosine",
            fp16                        = True, # Changed from FastLanguageModel.is_bfloat16_supported()
            bf16                        = False,      # Changed from FastLanguageModel.is_bfloat16_supported()
            logging_steps               = 20,
            eval_strategy               = "epoch",
            save_strategy               = "epoch",
            load_best_model_at_end      = True,
            metric_for_best_model       = "eval_loss",
            greater_is_better           = False,
            dataset_text_field          = "text",
            max_seq_length              = MAX_SEQ_LENGTH,
            packing                     = True,
            dataset_num_proc = 1,
            seed                        = SEED,
            remove_unused_columns = False,
        ),
    )

    trainer.train()

    # ── Quick eval on test split ───────────────────────────────────────────────
    test_metrics = trainer.state.log_history[-1]
    print(f"Fold {fold_idx + 1} val metrics: {test_metrics}")
    fold_results.append(test_metrics)

    # ── Save LoRA adapter ─────────────────────────────────────────────────────
    model.save_pretrained(f"{fold_output_dir}/lora_adapter")
    tokenizer.save_pretrained(f"{fold_output_dir}/lora_adapter")
    print(f"LoRA adapter saved to {fold_output_dir}/lora_adapter")

    # Free VRAM before next fold
    # del model, tokenizer, trainer
    import gc, torch
    gc.collect()
    torch.cuda.empty_cache()

    break


 FOLD 1 / 3
  train=573893  val=122978  test=122978
==((====))==  Unsloth 2026.3.17: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.3.17 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/10000 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=1):   0%|          | 0/10000 [00:00<?, ? examples/s]

...

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,990 | Num Epochs = 1 | Total steps = 312
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 1 x 1) = 16
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,0.512143,0.505475


Fold 1 val metrics: {'train_runtime': 1013.6221, 'train_samples_per_second': 4.923, 'train_steps_per_second': 0.308, 'total_flos': 1.1327421849329664e+16, 'train_loss': 0.6613271771333157, 'epoch': 1.0, 'step': 312}
LoRA adapter saved to ./outputs/fold_1/lora_adapter


In [21]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

try:
  del model, tokenizer, trainer
  import gc, torch
  gc.collect()
  torch.cuda.empty_cache()
except:
  print('a')
finally:
  gc.collect()
  torch.cuda.empty_cache()

a


### Eval

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"

SYSTEM_PROMPT = (
    "Your task is to extract address components from the user's message. "
    "Ignore any missing information and exclude those fields from your output. "
    "Return the data in JSON format using this structure:\n"
    "Output format:\n"
    "{\n"
    '  "house_number": "<house number>",\n'
    '  "street": "<street name>",\n'
    '  "city": "<city name>",\n'
    '  "postal_code": "<postal or zip code>",\n'
    '  "state": "<state or province name>",\n'
    '  "country": "<country name>"\n'
    "}"
)

bnb_config = BitsAndBytesConfig(load_in_4bit=True)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_config,
    device_map          = "auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = PeftModel.from_pretrained(base_model, "./outputs/fold_1/lora_adapter")
model.eval()

text = "PoLaND pawia 40d/6 52-235 abb"
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": text},
]

inputs         = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")
input_ids      = inputs["input_ids"].to("cuda")
attention_mask = inputs["attention_mask"].to("cuda")

with torch.no_grad():
    outputs = model.generate(
        input_ids      = input_ids,
        attention_mask = attention_mask,
        max_new_tokens = 128,
        do_sample      = False,
    )

response = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
print(response)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{"house_number": "40/6", "street": "Pawia", "city": "Połand", "country": "PL", "postal_code": "52-235", "state": "WY"}


In [ ]:
import os
from typing import List
import torch
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from data_preparation.preprocess_data import preprocess_data, make_splits, evaluate_predictions



def load_model_and_tokenizer(adapter_path: str):
    bnb_config = BitsAndBytesConfig(load_in_4bit=True)
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()
    return model, tokenizer


def batch_generate(texts: List[str], model, tokenizer, device: str = "cuda", batch_size: int = 8) -> List[str]:
    preds = []
    pad_id = getattr(tokenizer, "pad_token_id", 0) or 0

    for i in tqdm(range(0, len(texts), batch_size), desc="Generating"):
        batch = texts[i : i + batch_size]

        input_ids_list = []
        attention_list = []
        for text in batch:
            messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": text}]
            inp = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")
            ids = inp["input_ids"].squeeze(0)
            att = inp["attention_mask"].squeeze(0)
            input_ids_list.append(ids)
            attention_list.append(att)

        from torch.nn.utils.rnn import pad_sequence

        input_ids_padded = pad_sequence(input_ids_list, batch_first=True, padding_value=pad_id).to(device)
        attention_padded = pad_sequence(attention_list, batch_first=True, padding_value=0).to(device)

        with torch.no_grad():
            outputs = model.generate(
                input_ids=input_ids_padded,
                attention_mask=attention_padded,
                max_new_tokens=128,
                do_sample=False,
            )

        input_lens = [ids.shape[0] for ids in input_ids_list]
        for j, out in enumerate(outputs):
            gen = out[input_lens[j] :]
            resp = tokenizer.decode(gen, skip_special_tokens=True)
            preds.append(resp)

    return preds


def run_eval(adapter_path: str, batch_size: int = 8, max_examples: int = None):
    df = preprocess_data()

    for fold_idx, split in enumerate(make_splits(df)):
        print(f"\n=== Fold {fold_idx + 1} ===")
        test_df = split["test"]
        if max_examples is not None:
            test_df = test_df.iloc[:max_examples]

        adapter_dir = os.path.join(adapter_path)
        try:
            model, tokenizer = load_model_and_tokenizer(adapter_dir)
        except Exception as e:
            print(f"Failed to load model/adapter from {adapter_dir}: {e}")
            return

        device = "cuda" if torch.cuda.is_available() else "cpu"
        model.to(device)

        texts = test_df["input"].astype(str).tolist()

        preds = batch_generate(texts, model, tokenizer, device=device, batch_size=batch_size)

        golds = test_df["target"].tolist()
        metrics = evaluate_predictions(golds, preds)

        print("Evaluation metrics:")
        print(f"  examples: {metrics['counts']['examples']}")
        print(f"  exact_match: {metrics['exact_match']:.4f}")
        print(f"  overall_item_accuracy: {metrics['overall_item_accuracy']:.4f}")
        print("  per_field_accuracy:")
        for f, acc in metrics["per_field_accuracy"].items():
            print(f"    {f}: {acc:.4f}")



        # only evaluate first fold by default
        break

In [ ]:
run_eval(
    adapter_path="./outputs/fold_1/lora_adapter",
    batch_size=64,
    max_examples=64,
)